In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-large',
    # model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_KEY"),
    base_url=os.getenv("BASE_URL"),
)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3,
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_KEY")
)

INPUT_FILE = "../inputs/HD5L.txt"
DB_DIR = "../ChromaDB/db_hd5l"

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:

from libAgent.markdownSplitter import markdownTextSplitter
chunks = markdownTextSplitter(INPUT_FILE)

from libAgent.retriver import RetriverFactory

# Dense Retriever (MMR)
dense_retriever = RetriverFactory.createChromaRetriverMMR(embeddings=embeddings,dbPath=DB_DIR)

# Sparse Retriever (BM25)
bm25_retriever = RetriverFactory.createBM25RetrieverFromDocuments(chunks)


# Hybrid Retriever
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

In [12]:
res = dense_retriever.invoke("how to change Language  ?")

In [15]:
print( res[2].page_content)

According to the direction of Figure 3-2, press the keypad until hear a 'click' sound. Do not install the keypad from other directions or it will cause poor contact.  
Figure 3-2 Install keypad  
<!-- image -->  
There are two steps in Figure 3-3.  
First, press the hook of the keypad according to direction 1. Second, take out of the keypad according to direction 2.  
Figure 3-3 Dismantle keypad  
<!-- image -->  
<!-- image -->  
3


In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
                                PromptTemplate,
                                SystemMessagePromptTemplate,
                                HumanMessagePromptTemplate,
                                ChatPromptTemplate
                                )
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_classic import hub

prompt = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know.
    Answer in bullet points. Make sure your answer is relevant to the question and it is answered from the context only.
    Question: {question} 
    Context: {context} 
    Answer:
"""

template = ChatPromptTemplate.from_template(prompt)

def format_docs(docs):
    return '\n\n'.join([doc.page_content for doc in docs])

crg_chain = (
    RunnableParallel(
        {
            "context": hybrid_retriever | format_docs,
            "question": RunnablePassthrough()
        }
    )
    | template
    | llm
    | StrOutputParser()
)

In [17]:
res = crg_chain.invoke("how to change Language  ?")

print( res )

- Use the keypad menu to edit parameter F15.00 (Language selection). Navigate the 4-level menu: mode setting → function group → function parameter → parameter setting.
- Locate F15.00 under Group F15 and change its value: 0 = Chinese (default), 1 = English (2–9 unused).
- Save the new value (press the Enter/confirm key in the fourth-level menu).
- If a user password is set (F01.00 ≠ 00000), you must enter the correct password before you can change parameters; otherwise parameters are view-only.
